# ETL Guide — Movie Rental Data Warehouse

This notebook is a step-by-step guide to the ETL process that populates the **`movie_rental_dw`** Data Warehouse from the **Sakila** OLTP movie rental database.

**Reference:** This notebook implements the design described in `report.docx` and the schema in `sql/create_tables.sql`. The same logic runs as a single script in `etl_pandas.py`.

## What this ETL does

| Stage | Action |
|---|---|
| **E**xtract | Read 13 source tables from `sakila` using `pandas.read_sql` |
| **T**ransform | Build 8 dimensions and 3 facts using **NumPy** and **Pandas** |
| **L**oad | Truncate the DW (FK-safe) and bulk insert with `DataFrame.to_sql` |

## Target schema (Star Schema)

- **3 Fact tables:** `fact_rental`, `fact_payment`, `fact_return`
- **8 Dimension tables:** `dim_date`, `dim_customer`, `dim_film`, `dim_category`, `dim_language`, `dim_store`, `dim_staff`, `dim_location`

All three facts share the same **conformed dimensions** (date, customer, film, store, staff, location) so the DW supports cross-fact analysis.

## Tools used

- **Pandas** — DataFrames, joins, type conversion, SQL I/O
- **NumPy** — vectorized conditionals (`np.where`), clamping (`np.maximum`), explicit missing-value handling (`np.nan`)
- **SQLAlchemy + PyMySQL** — database connections

## ⚠️ Important: Run all cells in order

This notebook expects to be executed top-to-bottom. Running the fact-load cell without first running the dimension-load cells will fail with a foreign-key violation, because facts reference dimension surrogate keys. **Use `Cell → Run All` (or `Kernel → Restart & Run All`) for a clean run.**

## Prerequisites

1. MySQL Server 8.0 running on `localhost:3306`
2. `sakila` database loaded (run `mysql -u root < sql/fileDB.sql`)
3. `movie_rental_dw` schema created (run `mysql -u root < sql/create_tables.sql`)
4. Python packages installed (`pip install -r requirements.txt`)


---
## 1. Setup — Imports and Connections

We import both **Pandas** (DataFrames, SQL I/O) and **NumPy** (vectorized array operations). NumPy is used throughout the transform phase for fast conditional logic and explicit missing-value handling — both required by the assignment.

We create two SQLAlchemy engines:
- **`oltp_engine`** → reads from the source `sakila` database
- **`dw_engine`** → writes into the target `movie_rental_dw` database

The password is URL-encoded with `quote_plus` so special characters (`@`, `#`, `:`) in real passwords would not break the connection string.


In [1]:
import pandas as pd
import numpy as np
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

USER = "root"
PASSWORD = ""
HOST = "127.0.0.1"
PORT = "3306"
OLTP_DB = "sakila"
DW_DB = "movie_rental_dw"

ENCODED_PASSWORD = quote_plus(PASSWORD)
oltp_engine = create_engine(f"mysql+pymysql://{USER}:{ENCODED_PASSWORD}@{HOST}:{PORT}/{OLTP_DB}")
dw_engine = create_engine(f"mysql+pymysql://{USER}:{ENCODED_PASSWORD}@{HOST}:{PORT}/{DW_DB}")

# Sanity check: both connections work
with oltp_engine.connect() as c:
    print("OLTP version:", c.execute(text("SELECT VERSION()")).scalar())
with dw_engine.connect() as c:
    print("DW  version:", c.execute(text("SELECT VERSION()")).scalar())
print(f"Pandas: {pd.__version__}  |  NumPy: {np.__version__}")


OLTP version: 8.0.44
DW  version: 8.0.44
Pandas: 2.3.3  |  NumPy: 2.3.3


---
## 2. Extract — Read the OLTP tables

We load all source tables we need into pandas DataFrames in memory. The volumes are small (Sakila has only ~16k rentals), so we read them whole rather than streaming.

| Source table | Why we need it |
|---|---|
| `rental` | The rental events — source of `fact_rental` and `fact_return` |
| `payment` | The payment events — source of `fact_payment` |
| `inventory` | Bridge between rental and film (each rental references an inventory copy) |
| `customer`, `staff`, `store` | People/places involved in transactions |
| `film` | Film attributes (title, rating, duration, cost) |
| `address`, `city`, `country` | Location hierarchy (joined into a flat `dim_location`) |
| `category`, `film_category` | Film classification (one category per film in Sakila) |
| `language` | Language of each film |

We deliberately skip `actor`, `film_actor`, and `film_text` — they support full-text search and casting analysis, neither of which is in our business questions.


In [2]:
def load_table(name):
    return pd.read_sql(f"SELECT * FROM {name}", oltp_engine)

rental        = load_table("rental")
payment       = load_table("payment")
customer      = load_table("customer")
film          = load_table("film")
inventory     = load_table("inventory")
store         = load_table("store")
staff         = load_table("staff")
address       = load_table("address")
city          = load_table("city")
country       = load_table("country")
category      = load_table("category")
film_category = load_table("film_category")
language      = load_table("language")

extracted = {
    "rental": rental, "payment": payment, "customer": customer, "film": film,
    "inventory": inventory, "store": store, "staff": staff, "address": address,
    "city": city, "country": country, "category": category,
    "film_category": film_category, "language": language,
}
for name, df in extracted.items():
    print(f"{name:14} {len(df):>6} rows  |  cols: {list(df.columns)[:5]}...")


rental          16044 rows  |  cols: ['rental_id', 'rental_date', 'inventory_id', 'customer_id', 'return_date']...
payment         16044 rows  |  cols: ['payment_id', 'customer_id', 'staff_id', 'rental_id', 'amount']...
customer          599 rows  |  cols: ['customer_id', 'store_id', 'first_name', 'last_name', 'email']...
film             1000 rows  |  cols: ['film_id', 'title', 'description', 'release_year', 'language_id']...
inventory        4581 rows  |  cols: ['inventory_id', 'film_id', 'store_id', 'last_update']...
store               2 rows  |  cols: ['store_id', 'manager_staff_id', 'address_id', 'last_update']...
staff               2 rows  |  cols: ['staff_id', 'first_name', 'last_name', 'address_id', 'picture']...
address           603 rows  |  cols: ['address_id', 'address', 'address2', 'district', 'city_id']...
city              600 rows  |  cols: ['city_id', 'city', 'country_id', 'last_update']...
country           109 rows  |  cols: ['country_id', 'country', 'last_update']

**Quick preview** — what a raw OLTP rental row looks like before transformation:

In [3]:
rental.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


---
## 3. Transform — Build the dimensional model

We now reshape the normalized OLTP tables into a star schema using **NumPy and Pandas**.

### Where NumPy is used

| Pattern | Where | Why |
|---|---|---|
| `np.where(cond, a, b)` | `dim_customer`, `dim_staff` active_status | Vectorized 0/1 → string mapping (faster than `.apply(lambda)`) |
| `np.where(cond, 1, 0)` | `fact_return.late_return_flag` | Boolean → integer flag without `.astype(int)` |
| `np.maximum(x, 0)` | `fact_return.late_days` | Clamp negative deltas to zero (on-time returns) |
| `np.nan` + `pd.to_numeric(errors="coerce")` | `fact_payment.payment_amount` | Defensive cleaning before numeric aggregation |

### General strategy

1. **Surrogate keys** = natural keys (e.g., `customer_key = customer_id`). This is the simplest mapping and works because Sakila IDs are stable integers. For SCD Type 2 support we would generate new surrogate keys per change.
2. **Flat dimensions**: location info is denormalized into `dim_customer` and `dim_store` (just `city` and `country`) for query speed. A separate `dim_location` keeps the full address details for location-based analysis.
3. **Date key format**: `YYYYMMDD` integer (e.g., `20050524`). Easy to read, sorts correctly, joins fast.
4. **`fact_*` grain**: one row per transaction event (rental_id / payment_id / completed-rental_id).

We will build each dimension first, then the facts (which depend on the dimension keys).


### 3.1 `dim_date`

**Strategy:** Union of all date-bearing columns from the facts (`rental_date`, `return_date`, `payment_date`), then deduplicate and derive calendar attributes.

We use only actual event dates (not a continuous calendar from 2005–2006). This keeps `dim_date` small (~90 rows) and means every fact row finds its match. For trend analysis with gap-filling, a full calendar is better — but Sakila data is sparse so the event-driven approach is more honest.


In [4]:
rental_dates  = rental[["rental_date"]].rename(columns={"rental_date": "full_date"})
return_dates  = rental[["return_date"]].rename(columns={"return_date": "full_date"})
payment_dates = payment[["payment_date"]].rename(columns={"payment_date": "full_date"})

dim_date = pd.concat([rental_dates, return_dates, payment_dates], ignore_index=True)
dim_date = dim_date.dropna()                                        # drop NULL return_dates (films not yet returned)
dim_date["full_date"] = pd.to_datetime(dim_date["full_date"]).dt.date
dim_date = dim_date.drop_duplicates()

# Derive calendar attributes
dim_date["full_date_dt"] = pd.to_datetime(dim_date["full_date"])
dim_date["date_key"]       = dim_date["full_date_dt"].dt.strftime("%Y%m%d").astype(int)
dim_date["day_number"]     = dim_date["full_date_dt"].dt.day
dim_date["month_number"]   = dim_date["full_date_dt"].dt.month
dim_date["month_name"]     = dim_date["full_date_dt"].dt.month_name()
dim_date["quarter_number"] = dim_date["full_date_dt"].dt.quarter
dim_date["year_number"]    = dim_date["full_date_dt"].dt.year
dim_date["day_of_week"]    = dim_date["full_date_dt"].dt.day_name()

dim_date = dim_date[["date_key","full_date","day_number","month_number","month_name","quarter_number","year_number","day_of_week"]]
dim_date = dim_date.sort_values("date_key").reset_index(drop=True)

print(f"dim_date: {len(dim_date)} rows")
dim_date.head()


dim_date: 90 rows


,date_key,full_date,day_number,month_number,month_name,quarter_number,year_number,day_of_week
0,20050524,2005-05-24,24,5,May,2,2005,Tuesday
1,20050525,2005-05-25,25,5,May,2,2005,Wednesday
2,20050526,2005-05-26,26,5,May,2,2005,Thursday
3,20050527,2005-05-27,27,5,May,2,2005,Friday
4,20050528,2005-05-28,28,5,May,2,2005,Saturday


### 3.2 `dim_customer`

**Strategy:** customer + address + city + country → one denormalized row per customer.

- `first_name` + `last_name` → `customer_full_name` (single field for reports)
- `active` (0/1) → `active_status` ("Active"/"Inactive") using `np.where` — fully vectorized
- `address_id` joins through to get `city` and `country` directly on the customer row


In [5]:
customer_sel = customer[["customer_id","first_name","last_name","email","active","create_date","address_id"]]
address_sel  = address[["address_id","city_id"]]
city_sel     = city[["city_id","city","country_id"]]
country_sel  = country[["country_id","country"]]

df = customer_sel.merge(address_sel, on="address_id", how="left")
df = df.merge(city_sel, on="city_id", how="left")
df = df.merge(country_sel, on="country_id", how="left")

dim_customer = pd.DataFrame({
    "customer_key": df["customer_id"],
    "customer_id":  df["customer_id"],
    "customer_full_name": df["first_name"] + " " + df["last_name"],
    "email": df["email"],
    # np.where -> vectorized conditional, faster than df["active"].apply(lambda x: ...)
    "active_status": np.where(df["active"] == 1, "Active", "Inactive"),
    "create_date": pd.to_datetime(df["create_date"]).dt.date,
    "city": df["city"],
    "country": df["country"],
})

print(f"dim_customer: {len(dim_customer)} rows")
dim_customer.head()


dim_customer: 599 rows


,customer_key,customer_id,customer_full_name,email,active_status,create_date,city,country
0,1,1,MARY SMITH,MARY.SMITH@sakilacustomer.org,Active,2006-02-14,Sasebo,Japan
1,2,2,PATRICIA JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,Active,2006-02-14,San Bernardino,United States
2,3,3,LINDA WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,Active,2006-02-14,Athenai,Greece
3,4,4,BARBARA JONES,BARBARA.JONES@sakilacustomer.org,Active,2006-02-14,Myingyan,Myanmar
4,5,5,ELIZABETH BROWN,ELIZABETH.BROWN@sakilacustomer.org,Active,2006-02-14,Nantou,Taiwan


### 3.3 `dim_category` and `dim_language`

These are the smallest dimensions — direct 1:1 maps from the OLTP tables. The only transformation is renaming `name` to `category_name` / `language_name` for clarity.


In [6]:
dim_category = pd.DataFrame({
    "category_key": category["category_id"],
    "category_id":  category["category_id"],
    "category_name": category["name"],
})

dim_language = pd.DataFrame({
    "language_key": language["language_id"],
    "language_id":  language["language_id"],
    "language_name": language["name"],
})

print(f"dim_category: {len(dim_category)} rows  |  dim_language: {len(dim_language)} rows")
display(dim_category.head())
display(dim_language)


dim_category: 16 rows  |  dim_language: 6 rows


,category_key,category_id,category_name
0,1,1,Action
1,2,2,Animation
2,3,3,Children
3,4,4,Classics
4,5,5,Comedy


,language_key,language_id,language_name
0,1,1,English
1,2,2,Italian
2,3,3,Japanese
3,4,4,Mandarin
4,5,5,French
5,6,6,German


### 3.4 `dim_film`

**Strategy:** film + film_category → film attributes plus its category and language.

Sakila's `film_category` is technically many-to-many, but in practice each film has exactly one category. We still call `drop_duplicates(subset=["film_key"])` as a defensive guard against future data drift.

`category_key` and `language_key` on `dim_film` are foreign keys to `dim_category` and `dim_language` respectively — this is the textbook "outrigger" pattern, keeping the star schema clean.


In [7]:
df = film.merge(film_category, on="film_id", how="left")

dim_film = pd.DataFrame({
    "film_key": df["film_id"],
    "film_id":  df["film_id"],
    "title":    df["title"],
    "release_year":     df["release_year"],
    "rental_duration":  df["rental_duration"],
    "rental_rate":      df["rental_rate"],
    "length":           df["length"],
    "replacement_cost": df["replacement_cost"],
    "rating":           df["rating"],
    "category_key":     df["category_id"],
    "language_key":     df["language_id"],
})

dim_film = dim_film.drop_duplicates(subset=["film_key"])

print(f"dim_film: {len(dim_film)} rows")
dim_film.head()


dim_film: 1000 rows


,film_key,film_id,title,release_year,rental_duration,rental_rate,length,replacement_cost,rating,category_key,language_key
0,1,1,ACADEMY DINOSAUR,2006,6,0.99,86,20.99,PG,6,1
1,2,2,ACE GOLDFINGER,2006,3,4.99,48,12.99,G,11,1
2,3,3,ADAPTATION HOLES,2006,7,2.99,50,18.99,NC-17,6,1
3,4,4,AFFAIR PREJUDICE,2006,5,2.99,117,26.99,G,11,1
4,5,5,AFRICAN EGG,2006,6,2.99,130,22.99,G,8,1


### 3.5 `dim_store`

Same pattern as `dim_customer`: join through `address → city → country` to get a flat row per store.


In [8]:
store_sel   = store[["store_id","address_id"]]
address_sel = address[["address_id","address","city_id"]]
city_sel    = city[["city_id","city","country_id"]]
country_sel = country[["country_id","country"]]

df = store_sel.merge(address_sel, on="address_id", how="left")
df = df.merge(city_sel, on="city_id", how="left")
df = df.merge(country_sel, on="country_id", how="left")

dim_store = pd.DataFrame({
    "store_key": df["store_id"],
    "store_id":  df["store_id"],
    "store_address": df["address"],
    "city":    df["city"],
    "country": df["country"],
})

print(f"dim_store: {len(dim_store)} rows")
dim_store


dim_store: 2 rows


,store_key,store_id,store_address,city,country
0,1,1,47 MySakila Drive,Lethbridge,Canada
1,2,2,28 MySQL Boulevard,Woodridge,Australia


### 3.6 `dim_staff`

Similar to `dim_customer` but smaller. Uses the same `np.where` pattern for the `active_status` flag.


In [9]:
dim_staff = pd.DataFrame({
    "staff_key": staff["staff_id"],
    "staff_id":  staff["staff_id"],
    "staff_full_name": staff["first_name"] + " " + staff["last_name"],
    "email":      staff["email"],
    "active_status": np.where(staff["active"] == 1, "Active", "Inactive"),
    "store_id":   staff["store_id"],
})

print(f"dim_staff: {len(dim_staff)} rows")
dim_staff


dim_staff: 2 rows


,staff_key,staff_id,staff_full_name,email,active_status,store_id
0,1,1,Mike Hillyer,Mike.Hillyer@sakilastaff.com,Active,1
1,2,2,Jon Stephens,Jon.Stephens@sakilastaff.com,Active,2


### 3.7 `dim_location`

Covers **every** address in the OLTP system (customers, stores, and staff all link here). Kept separate from `dim_customer.city/country` because some analyses need the full address detail (district, postal code, phone) without joining customer-specific attributes.


In [10]:
address_sel = address[["address_id","address","district","city_id","postal_code","phone"]]
city_sel    = city[["city_id","city","country_id"]]
country_sel = country[["country_id","country"]]

df = address_sel.merge(city_sel, on="city_id", how="left")
df = df.merge(country_sel, on="country_id", how="left")

dim_location = pd.DataFrame({
    "location_key": df["address_id"],
    "address":   df["address"],
    "district":  df["district"],
    "city":      df["city"],
    "country":   df["country"],
    "postal_code": df["postal_code"],
    "phone":     df["phone"],
})

print(f"dim_location: {len(dim_location)} rows")
dim_location.head()


dim_location: 603 rows


,location_key,address,district,city,country,postal_code,phone
0,1,47 MySakila Drive,Alberta,Lethbridge,Canada,,
1,2,28 MySQL Boulevard,QLD,Woodridge,Australia,,
2,3,23 Workhaven Lane,Alberta,Lethbridge,Canada,,14033335568
3,4,1411 Lillydale Drive,QLD,Woodridge,Australia,,6172235589
4,5,1913 Hanoi Way,Nagasaki,Sasebo,Japan,35200,28303384290


---
## 4. Build the fact tables

Now that all dimensions are ready (in memory), we build the facts. Each fact:
1. Joins back to its OLTP source(s) to gather all foreign-key columns
2. Derives a `date_key` from the event date (YYYYMMDD format, matching `dim_date.date_key`)
3. Computes any measures (e.g., `payment_amount`, `late_days`) — using NumPy where vectorized math helps

`fact_return` is the most interesting of the three because it computes late-return measures from rental/return dates and the film's expected duration.


### 4.1 `fact_rental`

**Grain:** one row per rental transaction (`rental_id`).

**Join path:** `rental → inventory` (to get `film_id` and `store_id`) → `customer` (to get `address_id` for `location_key`).

**Measure:** `rental_count = 1` (additive — sum to get total rentals).


In [11]:
df = rental.merge(inventory, on="inventory_id", how="left")
df = df.merge(customer[["customer_id","address_id"]], on="customer_id", how="left")

fact_rental = pd.DataFrame({
    "rental_fact_key": df["rental_id"],
    "date_key":     pd.to_datetime(df["rental_date"]).dt.strftime("%Y%m%d").astype(int),
    "customer_key": df["customer_id"],
    "film_key":     df["film_id"],
    "store_key":    df["store_id"],
    "staff_key":    df["staff_id"],
    "location_key": df["address_id"],
    "rental_count": 1,
})

print(f"fact_rental: {len(fact_rental)} rows")
fact_rental.head()


fact_rental: 16044 rows


,rental_fact_key,date_key,customer_key,film_key,store_key,staff_key,location_key,rental_count
0,1,20050524,130,80,1,1,134,1
1,2,20050524,459,333,2,1,464,1
2,3,20050524,408,373,2,1,413,1
3,4,20050524,333,535,1,2,338,1
4,5,20050524,222,450,2,1,226,1


### 4.2 `fact_payment` — with NumPy-based amount cleaning

**Grain:** one row per payment (`payment_id`).

**Join path:** `payment → rental → inventory → film_id`, plus `customer → address_id`. This longer join chain is needed because payments link to films only through their rental.

**Defensive cleaning pattern (recommended by the professor):**

```python
df["amount"] = df["amount"].replace("", np.nan)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)
```

This three-step idiom:
1. `replace("", np.nan)` — empty strings become explicit missing values
2. `pd.to_numeric(errors="coerce")` — any non-numeric value also becomes NaN
3. `fillna(0)` — defaults missing amounts to zero (revenue queries never break)

Sakila's `amount` column happens to be clean, but using this pattern means the ETL stays correct if the source ever introduces dirty data.

**Measures:** `payment_amount` (revenue) and `payment_count = 1` (transaction count).


In [12]:
df = payment.merge(rental[["rental_id","inventory_id"]], on="rental_id", how="left")
df = df.merge(inventory, on="inventory_id", how="left")
df = df.merge(customer[["customer_id","address_id"]], on="customer_id", how="left")

# Defensive numeric cleaning using NumPy + Pandas
df["amount"] = df["amount"].replace("", np.nan)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0)

fact_payment = pd.DataFrame({
    "payment_fact_key": df["payment_id"],
    "date_key":     pd.to_datetime(df["payment_date"]).dt.strftime("%Y%m%d").astype(int),
    "customer_key": df["customer_id"],
    "film_key":     df["film_id"],
    "store_key":    df["store_id"],
    "staff_key":    df["staff_id"],
    "location_key": df["address_id"],
    "payment_amount": df["amount"],
    "payment_count":  1,
})

print(f"fact_payment: {len(fact_payment)} rows")
print(f"NULLs in film_key (payments without rentals): {fact_payment['film_key'].isna().sum()}")
print(f"Total revenue: ${fact_payment['payment_amount'].sum():,.2f}")
fact_payment.head()


fact_payment: 16044 rows
NULLs in film_key (payments without rentals): 0
Total revenue: $67,406.56


,payment_fact_key,date_key,customer_key,film_key,store_key,staff_key,location_key,payment_amount,payment_count
0,1,20050525,1,663,2,1,5,2.99,1
1,2,20050528,1,875,2,1,5,0.99,1
2,3,20050615,1,611,1,1,5,5.99,1
3,4,20050615,1,228,2,2,5,0.99,1
4,5,20050615,1,308,1,2,5,9.99,1


### 4.3 `fact_return` — NumPy-powered late-return detection

**Grain:** one row per **completed** rental (`return_date IS NOT NULL`). 183 rentals in Sakila have not yet been returned — they are correctly excluded.

**Date key:** uses `return_date` (not rental_date) because this fact answers *"when did films come back?"* questions.

**NumPy in action — the late-return math:**

```python
df["late_return_flag"] = np.where(actual > expected, 1, 0)   # vectorized 0/1 flag
df["late_days"]        = np.maximum(actual - expected, 0)    # clamp negatives to zero
```

- `np.where(cond, 1, 0)` is faster and clearer than `(cond).astype(int)`
- `np.maximum(x, 0)` is the standard NumPy idiom for "clamp at zero" — no `.apply(lambda)` needed

These measures enable the assignment business question *"Which films are returned late most often?"*.

**Derived measures:**
- `rental_duration_days` — actual days between rental and return
- `late_return_flag` — 1 if actual > film's expected duration, else 0
- `late_days` — how many days late (clamped to 0 for on-time returns)


In [13]:
df = rental.dropna(subset=["return_date"]).copy()
df = df.merge(inventory, on="inventory_id", how="left")
df = df.merge(film[["film_id","rental_duration"]], on="film_id", how="left")
df = df.merge(customer[["customer_id","address_id"]], on="customer_id", how="left")

df["rental_date"]          = pd.to_datetime(df["rental_date"])
df["return_date"]          = pd.to_datetime(df["return_date"])
df["rental_duration_days"] = (df["return_date"] - df["rental_date"]).dt.days

# --- NumPy vectorized late-return logic ---
# np.where  -> 0/1 flag without (bool).astype(int)
# np.maximum -> clamps negative deltas to 0 for on-time returns
df["late_return_flag"] = np.where(
    df["rental_duration_days"] > df["rental_duration"], 1, 0
)
df["late_days"] = np.maximum(
    df["rental_duration_days"] - df["rental_duration"], 0
)

fact_return = pd.DataFrame({
    "return_fact_key": df["rental_id"],
    "date_key":     df["return_date"].dt.strftime("%Y%m%d").astype(int),
    "customer_key": df["customer_id"],
    "film_key":     df["film_id"],
    "store_key":    df["store_id"],
    "staff_key":    df["staff_id"],
    "location_key": df["address_id"],
    "return_count": 1,
    "rental_duration_days": df["rental_duration_days"],
    "late_return_flag":     df["late_return_flag"],
    "late_days":            df["late_days"],
})

print(f"fact_return: {len(fact_return)} rows")
print(f"Late returns: {fact_return['late_return_flag'].sum()} ({fact_return['late_return_flag'].mean()*100:.1f}%)")
print(f"Average late days (when late): {fact_return.loc[fact_return['late_return_flag']==1, 'late_days'].mean():.1f}")
fact_return.head()


fact_return: 15861 rows
Late returns: 6403 (40.4%)
Average late days (when late): 2.6


,return_fact_key,date_key,customer_key,film_key,store_key,staff_key,location_key,return_count,rental_duration_days,late_return_flag,late_days
0,1,20050526,130,80,1,1,134,1,1,0,0
1,2,20050528,459,333,2,1,464,1,3,0,0
2,3,20050601,408,373,2,1,413,1,7,0,0
3,4,20050603,333,535,1,2,338,1,9,1,3
4,5,20050602,222,450,2,1,226,1,8,1,3


---
## 5. Load — Write to the Data Warehouse

### Load order matters (FK constraints)

`fact_*` tables have foreign keys to `dim_*` tables. So:

1. **Truncate** all DW tables (in reverse-FK order, with `FOREIGN_KEY_CHECKS=0`) — this makes the ETL **idempotent**: re-running gives a clean reload, not duplicates.
2. **Load dimensions first** (date → customer → category → language → film → store → staff → location)
3. **Load facts last** (rental → payment → return)

**Why this order matters:** every row in a fact table references dimension surrogate keys via foreign keys. If you try to load a fact before its dimensions exist, MySQL raises `IntegrityError: Cannot add or update a child row: a foreign key constraint fails`. The cells below MUST be executed in order.


In [14]:
def clear_dw_tables():
    """Truncate all DW tables in FK-safe order."""
    tables = [
        "fact_return", "fact_payment", "fact_rental",
        "dim_film", "dim_location", "dim_staff", "dim_store",
        "dim_language", "dim_category", "dim_customer", "dim_date",
    ]
    with dw_engine.begin() as conn:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
        for t in tables:
            conn.execute(text(f"TRUNCATE TABLE {t};"))
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    print("All DW tables truncated.")

clear_dw_tables()


All DW tables truncated.


### 5.1 Load dimensions

In [15]:
def load_to_dw(df, table_name):
    df.to_sql(table_name, dw_engine, if_exists="append", index=False, chunksize=1000)
    print(f"Loaded {len(df):>6} rows into {table_name}")

load_to_dw(dim_date,     "dim_date")
load_to_dw(dim_customer, "dim_customer")
load_to_dw(dim_category, "dim_category")
load_to_dw(dim_language, "dim_language")
load_to_dw(dim_film,     "dim_film")
load_to_dw(dim_store,    "dim_store")
load_to_dw(dim_staff,    "dim_staff")
load_to_dw(dim_location, "dim_location")


Loaded     90 rows into dim_date
Loaded    599 rows into dim_customer
Loaded     16 rows into dim_category
Loaded      6 rows into dim_language
Loaded   1000 rows into dim_film
Loaded      2 rows into dim_store


Loaded      2 rows into dim_staff


Loaded    603 rows into dim_location


### 5.2 Load facts

**Safety check first.** Before loading any fact, we verify that every dimension already has rows in the DW. If a dimension is empty (because Section 5.1 was skipped, or `clear_dw_tables()` ran without a subsequent dim reload), we abort with a clear message *before* MySQL raises a cryptic foreign-key error.


In [16]:
# Pre-flight check: every dimension must have rows before facts load.
# Without this guard, an out-of-order cell run produces:
#   IntegrityError 1452: Cannot add or update a child row: foreign key constraint fails
# which is much harder to diagnose than the explicit message below.
required_dims = ["dim_date","dim_customer","dim_category","dim_language",
                 "dim_film","dim_store","dim_staff","dim_location"]
dim_counts = {
    d: pd.read_sql(f"SELECT COUNT(*) AS n FROM {d}", dw_engine).iloc[0,0]
    for d in required_dims
}
empty = [d for d, n in dim_counts.items() if n == 0]
if empty:
    raise RuntimeError(
        f"Cannot load facts — these dimensions are empty: {empty}.\n"
        f"Run the dimension-load cell (Section 5.1) first, then re-run this cell."
    )
print("Pre-flight OK — all dimensions populated:")
for d, n in dim_counts.items():
    print(f"  {d:14} {n:>6} rows")

load_to_dw(fact_rental,  "fact_rental")
load_to_dw(fact_payment, "fact_payment")
load_to_dw(fact_return,  "fact_return")
print("\nETL completed successfully.")


Pre-flight OK — all dimensions populated:
  dim_date           90 rows
  dim_customer      599 rows
  dim_category       16 rows
  dim_language        6 rows
  dim_film         1000 rows
  dim_store           2 rows
  dim_staff           2 rows
  dim_location      603 rows


Loaded  16044 rows into fact_rental


Loaded  16044 rows into fact_payment


Loaded  15861 rows into fact_return

ETL completed successfully.


---
## 6. Verification

After loading, we check three things:
1. **Row counts** match expectations
2. **No FK orphans** (every fact row points to a valid dim row)
3. **Sample analytical queries** return sensible results


### 6.1 Row counts

In [17]:
counts_query = """
SELECT 'dim_date' AS tbl, COUNT(*) AS cnt FROM dim_date
UNION SELECT 'dim_customer', COUNT(*) FROM dim_customer
UNION SELECT 'dim_category', COUNT(*) FROM dim_category
UNION SELECT 'dim_language', COUNT(*) FROM dim_language
UNION SELECT 'dim_film',     COUNT(*) FROM dim_film
UNION SELECT 'dim_store',    COUNT(*) FROM dim_store
UNION SELECT 'dim_staff',    COUNT(*) FROM dim_staff
UNION SELECT 'dim_location', COUNT(*) FROM dim_location
UNION SELECT 'fact_rental',  COUNT(*) FROM fact_rental
UNION SELECT 'fact_payment', COUNT(*) FROM fact_payment
UNION SELECT 'fact_return',  COUNT(*) FROM fact_return
"""
pd.read_sql(counts_query, dw_engine)


,tbl,cnt
0,dim_date,90
1,dim_customer,599
2,dim_category,16
3,dim_language,6
4,dim_film,1000
5,dim_store,2
6,dim_staff,2
7,dim_location,603
8,fact_rental,16044
9,fact_payment,16044


### 6.2 FK integrity — orphan check

Every row in every fact should link to a real dimension row. If this returns anything > 0, the load is broken.


In [18]:
orphan_query = """
SELECT 'fact_rental orphan film' AS chk, COUNT(*) AS n
FROM fact_rental f LEFT JOIN dim_film d ON f.film_key = d.film_key WHERE d.film_key IS NULL
UNION
SELECT 'fact_payment orphan film', COUNT(*)
FROM fact_payment f LEFT JOIN dim_film d ON f.film_key = d.film_key WHERE d.film_key IS NULL AND f.film_key IS NOT NULL
UNION
SELECT 'fact_return orphan film', COUNT(*)
FROM fact_return f LEFT JOIN dim_film d ON f.film_key = d.film_key WHERE d.film_key IS NULL
"""
pd.read_sql(orphan_query, dw_engine)


,chk,n
0,fact_rental orphan film,0
1,fact_payment orphan film,0
2,fact_return orphan film,0


### 6.3 Business question samples

These four queries demonstrate that the DW answers real assignment business questions.


In [19]:
print("--- Q1. Top 10 most rented films ---")
display(pd.read_sql("""
    SELECT f.title, SUM(r.rental_count) AS total_rentals
    FROM fact_rental r JOIN dim_film f ON r.film_key = f.film_key
    GROUP BY f.title ORDER BY total_rentals DESC LIMIT 10
""", dw_engine))


--- Q1. Top 10 most rented films ---


,title,total_rentals
0,BUCKET BROTHERHOOD,34.0
1,ROCKETEER MOTHER,33.0
2,RIDGEMONT SUBMARINE,32.0
3,GRIT CLOCKWORK,32.0
4,SCALAWAG DUCK,32.0
5,JUGGLER HARDLY,32.0
6,FORWARD TEMPLE,32.0
7,HOBBIT ALIEN,31.0
8,ROBBERS JOON,31.0
9,ZORRO ARK,31.0


In [20]:
print("--- Q2. Top 10 revenue films ---")
display(pd.read_sql("""
    SELECT f.title, SUM(p.payment_amount) AS total_revenue
    FROM fact_payment p JOIN dim_film f ON p.film_key = f.film_key
    GROUP BY f.title ORDER BY total_revenue DESC LIMIT 10
""", dw_engine))


--- Q2. Top 10 revenue films ---


,title,total_revenue
0,TELEGRAPH VOYAGE,231.73
1,WIFE TURN,223.69
2,ZORRO ARK,214.69
3,GOODFELLAS SALUTE,209.69
4,SATURDAY LAMBS,204.72
5,TITANS JERK,201.71
6,TORQUE BOUND,198.72
7,HARRY IDAHO,195.70
8,INNOCENT USUAL,191.74
9,HUSTLER PARTY,190.78


In [21]:
print("--- Q3. Revenue by store ---")
display(pd.read_sql("""
    SELECT s.store_id, s.city, s.country, SUM(p.payment_amount) AS total_revenue
    FROM fact_payment p JOIN dim_store s ON p.store_key = s.store_key
    GROUP BY s.store_id, s.city, s.country ORDER BY total_revenue DESC
""", dw_engine))


--- Q3. Revenue by store ---


,store_id,city,country,total_revenue
0,2,Woodridge,Australia,33726.77
1,1,Lethbridge,Canada,33679.79


In [22]:
print("--- Q4. Most frequently late-returned films ---")
display(pd.read_sql("""
    SELECT f.title, SUM(r.late_return_flag) AS late_returns,
           ROUND(AVG(r.rental_duration_days),2) AS avg_duration,
           SUM(r.late_days) AS total_late_days
    FROM fact_return r JOIN dim_film f ON r.film_key = f.film_key
    GROUP BY f.title ORDER BY late_returns DESC, total_late_days DESC LIMIT 10
""", dw_engine))


--- Q4. Most frequently late-returned films ---


,title,late_returns,avg_duration,total_late_days
0,RIDGEMONT SUBMARINE,24.0,5.58,87.0
1,BUTTERFLY CHOCOLAT,23.0,5.37,77.0
2,TELEGRAPH VOYAGE,22.0,5.96,83.0
3,TIMBERLAND SKY,21.0,5.29,82.0
4,ROCKETEER MOTHER,20.0,4.94,75.0
5,GRIT CLOCKWORK,20.0,4.81,68.0
6,ENGLISH BULWORTH,20.0,4.60,61.0
7,CHANCE RESURRECTION,20.0,4.56,49.0
8,HUSTLER PARTY,19.0,5.95,70.0
9,PRINCESS GIANT,19.0,4.81,56.0


---
## 7. Summary

The Data Warehouse is now populated and ready for analytical reporting.

| Check | Result |
|---|---|
| Source rentals → fact_rental | 16,044 → 16,044 ✓ |
| Source payments → fact_payment | 16,044 → 16,044 ✓ |
| Returned rentals → fact_return | 15,861 (excludes 183 still out) ✓ |
| FK orphans | 0 ✓ |
| Business questions answerable | All 16 in the report ✓ |

### NumPy usage recap

| Location | NumPy call | Replaces |
|---|---|---|
| `dim_customer`, `dim_staff` | `np.where(active==1, "Active", "Inactive")` | `.apply(lambda x: ...)` |
| `fact_return.late_return_flag` | `np.where(actual > expected, 1, 0)` | `(actual > expected).astype(int)` |
| `fact_return.late_days` | `np.maximum(delta, 0)` | `delta.apply(lambda x: x if x > 0 else 0)` |
| `fact_payment.payment_amount` | `replace("", np.nan)` + `to_numeric(errors="coerce").fillna(0)` | unsafe direct assignment |

### How this ETL maps to the assignment requirements

- **Section 5.1 Extract:** Cell 5 reads all 13 OLTP tables
- **Section 5.2 Transform:** Cells 7-18 build dimensions + facts using NumPy + Pandas, derive surrogate keys, calculate late-return measures, handle missing return dates
- **Section 5.3 Load:** Cells 22-24 truncate (idempotency), then load dims-before-facts using `to_sql(if_exists='append')`. The fact-load cell has a pre-flight check that aborts cleanly if dimensions are empty.

### How to re-run

This notebook is fully idempotent — execute all cells from top to bottom (`Cell → Run All`). The `clear_dw_tables()` call ensures a clean slate on every run.

For production, you would typically:
- Replace `clear_dw_tables()` + full reload with incremental upserts based on `last_update` timestamps
- Add SCD Type 2 columns (`effective_from`, `effective_to`, `is_current`) to dimensions that change over time (customer, staff, film)
- Add data quality checks (validate `payment_amount >= 0`, etc.) before loading facts
